# 2.5 · Probability and statistics: reasoning about uncertainty

**Estimated time:** 35–45 minutes · **Prerequisite:** counting, fractions and running simple Python code.

### By the end, you should be able to

- Explain a **random experiment**, an **outcome**, and an **event**.
- Interpret **probability** as a number between 0 and 1.
- Compare **theoretical probability** with **observed relative frequency**.
- See why estimates become more stable as sample sizes grow (without expecting them to be perfectly equal).
- Compute a simple **conditional probability** and basic summaries such as the mean.

We first use coin tosses because we know the ideal mathematical probabilities. Then we return to our **four fictional learners** and show why tiny datasets require caution.


## 1. Probability is about uncertain outcomes

Toss a fair coin. There are two possible outcomes: **Heads (H)** and **Tails (T)**. A fair coin assigns each outcome probability `1/2 = 0.5 = 50%`.

An **event** is a set of outcomes we care about. If we toss a coin *twice*, the four equally likely outcomes are HH, HT, TH and TT. The event “exactly one head” includes **HT and TH**, so:

$$
P(\text{exactly one head}) = \frac{1}{4} + \frac{1}{4} = \frac{1}{2}.
$$

These calculations assume **fair and independent** tosses. If the coin is biased, the probabilities change.


### What, why, how, and where?

**What:** Probability quantifies how likely an event is under an assumed process or model. A probability of 0 means impossible under that model; 1 means certain.

**Why:** predictions and real-world observations can be uncertain even when we have good data.

**How:** define the experiment and event first. With two independent fair coin tosses, there are four equally likely outcomes: HH, HT, TH, TT. The event "exactly one head" is `{HT, TH}`, so its probability is `2/4 = 0.5`.

**ML connection:** probability helps us interpret prediction uncertainty and model evaluation, but an assumption like "fair independent tosses" must always be stated.


<picture>
  <source type="image/svg+xml" srcset="images/17-coin-outcomes.svg">
  <img src="images/17-coin-outcomes.png" alt="A two-toss probability tree showing all four outcomes and probability one-quarter for each" loading="lazy">
</picture>

*Figure: A two-toss probability tree showing all four outcomes and probability one-quarter for each.*


**Interpretation:** A probability of `0.5` does not promise that the first 10 tosses will produce *exactly* five heads. It describes the long-run behavior of the ideal random process, not a guaranteed count in a small sample.

## 2. Observed frequency is an estimate

We can *simulate* tossing a fair coin. Each trial produces either `0` (tails) or `1` (heads). The **relative frequency** of heads is:

$$
\frac{\text{number of observed heads}}{\text{total tosses}}.
$$

This is an **estimate** from the results we observed. It need not equal the theoretical probability exactly, even with many tosses.

We use a fixed random seed so the outputs are reproducible on different learners' machines.


### Theoretical probability is not an observed count

For a fair coin, `P(heads)=0.5` describes the generating process. An observed fraction—for example, 7 heads in 10 tosses, or `7/10=0.7`—describes one **sample**. The two can differ without any contradiction.

**Why more data helps:** the running fraction tends to settle *near* 0.5 as we collect many independent fair tosses; it need not hit 0.5 exactly. A larger sample reduces the influence of any one unusual result, but the correctness of an estimate still depends on assumptions and sampling.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
tosses = rng.integers(0,2,size=1000)  # 0 = tails, 1 = heads

for n in [10,100,1000]:
    heads = int(tosses[:n].sum())
    print(f'First {n:4} tosses: {heads:4} heads; fraction = {heads/n:.3f}')


Now look at the **complete running estimate**, not just the three snapshots. The orange line shows the theoretical probability 0.5, while the teal line records the fraction observed as we collect more tosses.


<picture>
  <source type="image/svg+xml" srcset="images/18-sample-size.svg">
  <img src="images/18-sample-size.png" alt="A reproducible plot of the running observed fraction of heads across 1000 simulated coin tosses, with a 0.5 reference line" loading="lazy">
</picture>

*Figure: A reproducible plot of the running observed fraction of heads across 1000 simulated coin tosses, with a 0.5 reference line.*


In [ ]:
running_fraction = np.cumsum(tosses) / np.arange(1,len(tosses)+1)
fig,ax=plt.subplots(figsize=(8,3.7))
ax.plot(np.arange(1,len(tosses)+1),running_fraction,label='Observed fraction')
ax.axhline(.5,linestyle='--',label='Fair coin probability (0.5)')
ax.set(xlabel='Toss count',ylabel='Fraction of heads',title='Try changing the random seed or number of tosses')
ax.legend();plt.show()


### What larger samples really mean

Larger samples **usually** make an empirical proportion more stable, because a few unusual outcomes have less influence. This is an example of the **law of large numbers**. However:

- The proportions do **not** become exactly equal to 0.5 after a fixed number of tosses.
- The actual **counts** of heads and tails do **not** need to be equal.
- The process must be appropriate for the assumptions: a biased coin will stabilize around its own probability, not 0.5.

This matters in ML because a model's evaluation score on a **small test set** can be noisy.


## 3. A conditional probability asks a more specific question

Return to the toy class:

| Learner | Hours | Score | Passed (score ≥ 6)? |
|:--|--:|--:|:--|
| Alice | 1 | 3 | No |
| Ben | 2 | 5 | No |
| Chika | 3 | 7 | Yes |
| Dara | 4 | 9 | Yes |

Across all four learners, the **observed fraction** who passed is `2/4 = 0.5`. But perhaps we ask: **Among learners who studied at least 3 hours, what fraction passed?**

Only Chika and Dara are in that group, and both passed. So **`P(pass | Hours ≥ 3)`**, as *estimated from this toy sample*, is `2/2 = 1.0`.

The vertical bar `|` reads **“given that.”** It restricts which rows we count in the denominator.


### Conditional probability changes which rows we count

A conditional probability restricts attention to cases where some condition is true. Its core rule is `P(A | B) = P(A and B) / P(B)`, provided `P(B) > 0`. In a dataset, you can think of this as:

> Among **only the rows where B is true**, what fraction also satisfy A?

In our four-learner sample, `B` = "studied at least 3 hours" includes Chika and Dara (two rows). `A` = "passed" is true for both, so `P(pass | Hours≥3)` is estimated as `2/2=1` **in this invented sample**. It does not establish that studying caused passing. If no rows meet B, the observed fraction is undefined rather than zero.


<picture>
  <source type="image/svg+xml" srcset="images/19-conditional-probability.svg">
  <img src="images/19-conditional-probability.png" alt="A fully labeled count table for pass versus not pass grouped by study hours, including row/column totals" loading="lazy">
</picture>

*Figure: A fully labeled count table for pass versus not pass grouped by study hours, including row/column totals.*


In [ ]:
import pandas as pd

sample = pd.DataFrame({
    'Learner':['Alice','Ben','Chika','Dara'],
    'Hours':[1,2,3,4],
    'Score':[3,5,7,9],
})
sample['Passed'] = sample['Score']>=6
sample['Hours_3_or_more'] = sample['Hours']>=3

print('Overall pass fraction:',sample['Passed'].mean())
print('Pass fraction given Hours>=3:',
      sample.loc[sample['Hours_3_or_more'],'Passed'].mean())
print('\nCounts:\n',pd.crosstab(sample['Passed'],sample['Hours_3_or_more'],margins=True))


**Critical caution:** These are only **four invented observations**. The conditional estimate `1.0` is not a law about students, a reliable estimate for the wider world, or proof that studying three hours *causes* a pass. Statistical relationships can reflect other differences between groups. The difference between **observation** and **causal explanation** will matter in later ML and AI safety evaluation.

## 4. Mean and spread: two basic statistical summaries

A **mean** (average) tells us the center of a group of numbers. For our scores `[3,5,7,9]`, the mean is `(3+5+7+9)/4 = 6`.

But the mean hides **variation**: learners scored differently. Standard deviation gives one way to summarize **spread**. For this beginner course, you only need to understand that a larger standard deviation usually means the values vary more around the mean. We'll use PyTorch to calculate it.

Be careful when interpreting a statistic: a single mean or pass rate is not a complete description of a dataset.


### Same mean, different variability

Two classes might have scores `[5,5,5]` and `[3,5,7]`. Both have mean 5, but the first has **no spread** while the second varies. Standard deviation summarizes typical variation around the mean (in the original units).

**Why this matters:** reporting only average model accuracy can conceal unreliable behavior on some examples. Also check variability, subgroup differences, and sample size; a single statistic never tells the whole story.


In [ ]:
scores = torch.tensor([3.,5.,7.,9.])
print('Average score:',scores.mean().item())
print('Minimum and maximum:',scores.min().item(),scores.max().item())
print('Population standard deviation:',round(scores.std(unbiased=False).item(),3))

second_group=torch.tensor([6.,6.,6.,6.])
print('Another group with same mean:',second_group.mean().item())
print('But different spread:',second_group.std(unbiased=False).item())


## 5. Why probability matters for machine learning and AI safety

Models make predictions on imperfect observations, under uncertainty. When evaluating models, we often report fractions such as **accuracy** or **failure rate**. These numbers depend on which examples were tested, how many examples were included, and what we counted as success.

Even if two models have the same accuracy on a tiny set, that alone does not show they have the same reliability across different situations. We need to be able to discuss **sampling, uncertainty, and distribution of outcomes** rather than treating one result as absolute certainty.

The purpose of this lesson is *not* to master all of statistics. It is to develop the habits of asking: “What happened?”, “How often?”, “Among whom?”, and “How confident should we be?”

## 6. Small experiment: compare different random seeds

Re-run the following cell using seed `0`, `1`, and `2`. Do the first 10 flips always have the same fraction of heads? Do the first 1000 fractions become identical?


In [ ]:
for seed in [0,1,2]:
    flips = np.random.default_rng(seed).integers(0,2,size=1000)
    print(f'seed {seed}: first 10 = {flips[:10].mean():.2f}, all 1000 = {flips.mean():.3f}')


### Check your understanding

1. What is the difference between theoretical probability and the **fraction observed** in a sample?
2. For two fair independent tosses, what is the probability of **at least one head**?
3. Why should you not expect **exactly** 500 heads in every set of 1,000 fair-coin tosses?
4. What does `P(pass | Hours ≥ 3)` mean—and which learners are in its denominator?
5. Two groups have the same mean score. Does that guarantee that they have the same spread?

**Takeaway:** Probability gives a way to reason about uncertain events; statistics helps us describe the outcomes that actually occurred. Both are essential for interpreting a model's behavior responsibly.

➡️ **Next:** [Chapter 2 checkpoint](quizzes/chapter-02-checkpoint.ipynb). Chapter 3 (Machine Learning Basics) is the next planned chapter.
